# 🦀 Day 7: Mini-project — High-performance log parser
---

Today we build a **high-performance log parser** — applying Week 4 techniques.

- **Log format**: `[2026-01-15 14:30:45] [WARN] Connection timeout from 192.168.1.1`
- **Zero-copy parsing**: Use `&str` slices, avoid allocations
- **Struct**: `LogEntry { timestamp, level, message, source_ip }`
- **Streaming**: Process line by line, don't load all into memory
- **Performance**: Pre-compiled regex, buffered I/O, avoid String

In [ ]:
:dep regex = "1"

In [ ]:
use regex::Regex;
use std::collections::HashMap;

/// Log format: [2026-01-15 14:30:45] [WARN] Message from 192.168.1.1
#[derive(Debug)]
struct LogEntry<'a> {
    timestamp: &'a str,
    level: &'a str,
    message: &'a str,
    source_ip: Option<&'a str>,
}

fn parse_log_line(re: &Regex, line: &str) -> Option<LogEntry<'_>> {
    let cap = re.captures(line)?;
    Some(LogEntry {
        timestamp: cap.name("ts")?.as_str(),
        level: cap.name("level")?.as_str(),
        message: cap.name("msg")?.as_str(),
        source_ip: cap.name("ip").map(|m| m.as_str()),
    })
}

// Pre-compile regex once (avoids re-parsing on every line)
let re = Regex::new(r"\[(?P<ts>[^\]]+)\] \[(?P<level>\w+)\] (?P<msg>.+?)(?: from (?P<ip>[\d.]+))?$").unwrap();

let lines = [
    "[2026-01-15 14:30:45] [WARN] Connection timeout from 192.168.1.1",
    "[2026-01-15 14:30:46] [ERROR] Database connection failed",
    "[2026-01-15 14:30:47] [INFO] Server started",
];

let mut by_level: HashMap<&str, usize> = HashMap::new();
for line in &lines {
    if let Some(entry) = parse_log_line(&re, line) {
        *by_level.entry(entry.level).or_insert(0) += 1;
        println!("{:?}", entry);
    }
}
println!("By level: {:?}", by_level);

## Streaming & BufReader

For large files, use `BufReader` and process line by line:
```rust
use std::io::BufRead;
let file = File::open("logs.txt")?;
for line in BufReader::new(file).lines() {
    let line = line?;
    if let Some(entry) = parse_log_line(&re, &line) { /* ... */ }
}
```

**rayon** (Month 4) can parallelize across chunks for even faster processing.

In [ ]:
// Statistics: count by level, most frequent IPs
use std::collections::HashMap;

let re = regex::Regex::new(r"\[(?P<ts>[^\]]+)\] \[(?P<level>\w+)\] (?P<msg>.+?)(?: from (?P<ip>[\d.]+))?$").unwrap();
let lines = [
    "[2026-01-15 14:30:45] [WARN] Connection timeout from 192.168.1.1",
    "[2026-01-15 14:30:46] [ERROR] Database connection failed",
    "[2026-01-15 14:30:47] [WARN] Retry from 192.168.1.1",
];

let mut by_level: HashMap<&str, usize> = HashMap::new();
let mut by_ip: HashMap<&str, usize> = HashMap::new();

for line in &lines {
    if let Some(cap) = re.captures(line) {
        let level = cap.name("level").unwrap().as_str();
        *by_level.entry(level).or_insert(0) += 1;
        if let Some(ip) = cap.name("ip") {
            *by_ip.entry(ip.as_str()).or_insert(0) += 1;
        }
    }
}
println!("By level: {:?}", by_level);
println!("By IP: {:?}", by_ip);

## 📝 Exercise: Add filtering by date range or log level

Extend the parser to support:
- Filter entries by log level (e.g. only WARN and ERROR)
- Filter by date range (parse timestamp, compare)

Use `chrono` for date parsing in a real project; for EvCxR, string comparison may suffice.

In [ ]:
// YOUR CODE HERE
// fn filter_by_level(entries: &[LogEntry], levels: &[&str]) -> Vec<&LogEntry>
// fn filter_by_date(entries: &[LogEntry], from: &str, to: &str) -> Vec<&LogEntry>

## 🎯 Key Takeaways

1. **Zero-copy parsing** with `&str` slices avoids allocations
2. **Pre-compiled regex** is critical for performance
3. **BufReader** + line-by-line processing keeps memory low for large files
4. **LogEntry** with lifetime `'a` keeps references into the source string
5. **rayon** can parallelize log processing for large datasets
6. **chrono** for date parsing in production projects

---
**🎉 Month 5 Complete! Next: Month 6 — Capstone & Mastery!**